# Control secuencial y memoria : método en la práctica

**Nivel:** 2  
**Requisitos previos:** Control secuencial y memoria nivel 1  
**Tiempo de lectura:** 25 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. Las observaciones demoradas, el estado cambiante de trabajos y las consecuencias de las acciones hacen que el control de software sea secuencial. La matriz distingue imitación recurrente, bandido contextual miope, PPO con máscara y búsqueda acotada con modelo aprendido, sin agruparlos como una ‘IA’ genérica.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/benchmark-adequacy.json`
- `data/derived/episodes/M05-C07-nominal-40000.json`
- `data/derived/episodes/M06-C07-nominal-40000.json`
- `data/derived/episodes/M07-C07-nominal-40000.json`
- `data/derived/episodes/M08-C07-nominal-40000.json`

## Método

Lee la porción C07 de cancelación, inspecciona el éxito por variante y la secuencia guardada de decisiones y efectos. Las trazas muestran qué acción del controlador se ejecutó y si el verificador independiente aceptó el resultado. El notebook usa denominadores por episodio y mantiene visibles familias y semillas.

El RL secuencial optimiza \(G_t=\sum_{k=0}^{T-t}\gamma^k r_{t+k}\); una política con transición aprendida usa \(\hat s_{t+1}=f_\phi(s_t,a_t)\) para puntuar una búsqueda acotada. Son métodos distintos del paquete, M07 y M08, con objetivos diferentes y no equivalentes a un bandido de un paso.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M05', 'M06', 'M07', 'M08']
sample_case = 'C07'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C07 | seed: 40000
Methods: M05, M06, M07, M08


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M05 | action= a-cancel | reason= recurrent_behavior_cloning | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M06 | action= a-cancel | reason= linucb_myopic | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M07 | action= a-cancel | reason= trained_nonrecurrent_maskable_ppo | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M08 | action= a-cancel | reason= learned_transition_bounded_search | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


### Ejecuta las políticas publicadas y compara la corrida

In [4]:
# Los resultados de planificación se leen de las trazas congeladas; no hay llamada implícita a un LLM.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M05 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M06 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M07 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M08 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}


### Compara la secuencia completa y no solo la primera acción

In [5]:
# La igualdad de secuencia es diagnóstica; la verificación independiente decide el éxito.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M05 a-cancel | terminal= succeeded | verified= 1
M06 a-cancel | terminal= succeeded | verified= 1
M07 a-cancel | terminal= succeeded | verified= 1
M08 a-cancel | terminal= succeeded | verified= 1


## Resultados

Los 40 fallos centrales de M12 están en C07, mientras varios métodos tienen éxito en los 200 episodios C07. El resultado identifica una brecha concreta; el éxito agregado no establece si la causó la recurrencia, la máscara de acciones, el error del modelo o el tiempo de cancelación.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

La historia y las consecuencias futuras pueden importar, pero la complejidad no demuestra que la memoria sea útil. Lee juntas la traza y la tarea verificada; inspecciona las variantes fallidas antes de proponer un mecanismo nuevo.

## Referencias

- [GRU](https://arxiv.org/abs/1406.1078)
- [PPO](https://arxiv.org/abs/1707.06347)
- [LINUCB](https://arxiv.org/abs/1003.0146)